In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install open3d

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 140.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.0/228.0 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 89.9 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: werkzeug
    Found existing installation: Werkzeug 3.1.3
    Uninstalling Werkzeug-3.1.3:
      Successfully uninstalled Werkzeug-3.1.3
  Attempting uninstall: flask
    Found existing installation: Flask 3.1.0
    Un

In [ ]:
!pip install pycolmap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 106.7 MB/s eta 0:00:00


In [ ]:
# drive_image_dir = '/content/drive/MyDrive/dataset_monstree-master/mini3'
# drive_output_dir = '/content/drive/MyDrive/output'
# local_output_dir = '/content/output4'

# import os
# import pycolmap
# import open3d as o3d
# import numpy as np

# os.makedirs(local_output_dir, exist_ok=True)
# database_path = os.path.join(local_output_dir, "database.db")

# # діставання та метчинг фіч
# print("[INFO] Extracting features and matching...")
# pycolmap.extract_features(database_path=database_path, image_path=drive_image_dir)
# pycolmap.match_exhaustive(database_path=database_path)

# # SfM
# print("[INFO] Running incremental mapping...")
# reconstructions = pycolmap.incremental_mapping(
#     database_path=database_path,
#     image_path=drive_image_dir,
#     output_path=local_output_dir
# )

# first = next(iter(reconstructions.values()))

# # витягуємо точки та кольори
# points = []
# colors = []
# for pt in first.points3D.values():
#     points.append(pt.xyz)
#     colors.append([c / 255.0 for c in pt.color])  # нормалізуємо до [0, 1]

# # створюємо Open3D point cloud та зберігаємо
# pcd = o3d.geometry.PointCloud()
# pcd.points = o3d.utility.Vector3dVector(np.array(points))
# pcd.colors = o3d.utility.Vector3dVector(np.array(colors))
# ply_path = os.path.join(drive_output_dir, "sparse_reconstruction_4.ply")
# o3d.io.write_point_cloud(ply_path, pcd)

# print(f"[DONE] Sparse point cloud saved to: {ply_path}")

In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import open3d as o3d
from scipy.spatial import cKDTree
from scipy.sparse import lil_matrix
from scipy.optimize import least_squares
import pycolmap

# Define paths
drive_image_dir = '/content/drive/MyDrive/3D_reconstruction/mini3'
drive_output_dir = '/content/drive/MyDrive/3D_reconstruction/output'
local_output_dir = '/content/output4'

os.makedirs(local_output_dir, exist_ok=True)
database_path = os.path.join(local_output_dir, "database.db")

# SIFT
def manual_sift_detector(image, n_octaves=4, n_scales=3, contrast_threshold=0.04, edge_threshold=10):
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image

    # scale space pyramid
    sigma_init = 1.6
    k = 2**(1/n_scales)
    sigma_list = [sigma_init * (k**i) for i in range(n_scales + 3)]

    # Gaussian pyramids
    gaussian_pyramids = []
    for octave in range(n_octaves):
        octave_pyramids = []
        base_img = gray
        if octave > 0:
            # downsample
            base_img = cv2.resize(gray, (gray.shape[1] // (2**octave), gray.shape[0] // (2**octave)))

        # scale space for this octave
        for sigma in sigma_list:
            blur_img = cv2.GaussianBlur(base_img, (0, 0), sigmaX=sigma)
            octave_pyramids.append(blur_img)

        gaussian_pyramids.append(octave_pyramids)

    # detect keypoints by finding extrema in DoG space
    keypoints = []

    for octave in range(n_octaves):
        dog_images = []
        for i in range(len(gaussian_pyramids[octave]) - 1):
            dog = gaussian_pyramids[octave][i+1].astype(np.float32) - gaussian_pyramids[octave][i].astype(np.float32)
            dog_images.append(dog)

        # find extrema in DoG
        for i in range(1, len(dog_images) - 1):
            current_dog = dog_images[i]
            h, w = current_dog.shape

            # search of local extrema
            for y in range(1, h - 1):
                for x in range(1, w - 1):
                    center_val = current_dog[y, x]

                    if abs(center_val) < contrast_threshold:
                        continue

                    is_extrema = True

                    # check if it's greater than all neighbors or less than all neighbors
                    if center_val > 0:  # check for maximum
                        for dy in [-1, 0, 1]:
                            for dx in [-1, 0, 1]:
                                for di in [-1, 0, 1]:
                                    if dy == 0 and dx == 0 and di == 0:
                                        continue

                                    # get neighbor in 3D space
                                    if (i + di < 0 or i + di >= len(dog_images) or
                                        y + dy < 0 or y + dy >= h or
                                        x + dx < 0 or x + dx >= w):
                                        continue

                                    if center_val <= dog_images[i + di][y + dy, x + dx]:
                                        is_extrema = False
                                        break

                                if not is_extrema:
                                    break

                            if not is_extrema:
                                break
                    else:  # check for minimum
                        for dy in [-1, 0, 1]:
                            for dx in [-1, 0, 1]:
                                for di in [-1, 0, 1]:
                                    if dy == 0 and dx == 0 and di == 0:
                                        continue

                                    # get neighbor in 3D space
                                    if (i + di < 0 or i + di >= len(dog_images) or
                                        y + dy < 0 or y + dy >= h or
                                        x + dx < 0 or x + dx >= w):
                                        continue

                                    if center_val >= dog_images[i + di][y + dy, x + dx]:
                                        is_extrema = False
                                        break

                                if not is_extrema:
                                    break

                            if not is_extrema:
                                break

                    if is_extrema:
                        # edge response test
                        Dxx = current_dog[y, x+1] + current_dog[y, x-1] - 2*center_val
                        Dyy = current_dog[y+1, x] + current_dog[y-1, x] - 2*center_val
                        Dxy = ((current_dog[y+1, x+1] - current_dog[y+1, x-1]) -
                              (current_dog[y-1, x+1] - current_dog[y-1, x-1])) / 4.0

                        # simplified Harris corner response
                        trace = Dxx + Dyy
                        det = Dxx * Dyy - Dxy * Dxy

                        if det == 0:
                            continue

                        r = trace**2 / det
                        if r > (edge_threshold + 1)**2 / edge_threshold:
                            continue

                        # keypoint information (x, y, octave, scale)
                        scale_factor = 2**octave
                        kp = cv2.KeyPoint(
                            x=x * scale_factor,
                            y=y * scale_factor,
                            size=sigma_list[i] * scale_factor,
                            angle=-1,  # orientation will be computed later
                            response=abs(center_val),
                            octave=octave,
                            class_id=-1
                        )
                        keypoints.append(kp)

    return keypoints

def compute_sift_descriptors(image, keypoints, n_bins=8, n_hist=4, desc_factor=3):
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image

    # precompute gradients
    dx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=1)
    dy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=1)

    # compute gradient magnitude and orientation
    magnitude = np.sqrt(dx**2 + dy**2)
    orientation = np.arctan2(dy, dx)
    orientation = np.rad2deg(orientation) % 360

    # for each keypoint, compute dominant orientation
    for i, kp in enumerate(keypoints):
        x, y = int(kp.pt[0]), int(kp.pt[1])
        scale = kp.size / desc_factor

        # window for orientation assignment
        radius = int(scale * 6)
        radius = max(1, radius)

        x_min = max(0, x - radius)
        x_max = min(gray.shape[1] - 1, x + radius)
        y_min = max(0, y - radius)
        y_max = min(gray.shape[0] - 1, y + radius)

        # extract region of interest
        window_mag = magnitude[y_min:y_max+1, x_min:x_max+1]
        window_ori = orientation[y_min:y_max+1, x_min:x_max+1]

        # orientation histogram
        hist = np.zeros(36)

        for j in range(window_mag.shape[0]):
            for k in range(window_mag.shape[1]):
                bin_idx = int(window_ori[j, k] // 10) % 36
                weight = window_mag[j, k] * np.exp(
                    -((j + y_min - y)**2 + (k + x_min - x)**2) / (2 * scale**2)
                )
                hist[bin_idx] += weight

        # find dominant orientation
        max_bin = np.argmax(hist)
        kp.angle = max_bin * 10.0

    # compute descriptors
    descriptors = []

    for kp in keypoints:
        x, y = int(kp.pt[0]), int(kp.pt[1])
        scale = kp.size / desc_factor
        theta = np.deg2rad(kp.angle)

        # window for descriptor computation
        radius = int(n_hist * scale * np.sqrt(2) / 2)
        radius = max(1, radius)

        # skip keypoints too close to the border
        if (x - radius < 0 or x + radius >= gray.shape[1] or
            y - radius < 0 or y + radius >= gray.shape[0]):
            continue

        descriptor = np.zeros((n_hist, n_hist, n_bins))

        # rotation matrix for aligning with dominant orientation
        cos_theta = np.cos(theta)
        sin_theta = np.sin(theta)

        # compute descriptor over spatial bins
        for dy in range(-radius, radius + 1):
            for dx in range(-radius, radius + 1):
                # rotate coordinates
                rot_dx = cos_theta * dx - sin_theta * dy
                rot_dy = sin_theta * dx + cos_theta * dy

                # determine which spatial bin this sample falls into
                bin_x = (rot_dx / scale) + n_hist / 2 - 0.5
                bin_y = (rot_dy / scale) + n_hist / 2 - 0.5

                if bin_x < -1 or bin_x >= n_hist or bin_y < -1 or bin_y >= n_hist:
                    continue

                # sample point in the image
                sample_x = x + dx
                sample_y = y + dy

                # gradient at this point
                sample_mag = magnitude[sample_y, sample_x]
                sample_ori = orientation[sample_y, sample_x]

                # rotate orientation relative to keypoint orientation
                rot_ori = (sample_ori - kp.angle) % 360

                # determine which orientation bin this sample falls into
                ori_bin = rot_ori / (360.0 / n_bins)

                # weighting by distance from center
                weight = sample_mag * np.exp(-(dx**2 + dy**2) / (2 * (0.5 * n_hist * scale)**2))

                # trilinear interpolation
                for bin_i in range(2):
                    bin_yi = int(np.floor(bin_y) + bin_i)
                    if bin_yi < 0 or bin_yi >= n_hist:
                        continue

                    y_weight = 1.0 - abs(bin_y - (bin_yi + 0.5))

                    for bin_j in range(2):
                        bin_xi = int(np.floor(bin_x) + bin_j)
                        if bin_xi < 0 or bin_xi >= n_hist:
                            continue

                        x_weight = 1.0 - abs(bin_x - (bin_xi + 0.5))

                        for bin_k in range(2):
                            bin_oi = int(np.floor(ori_bin) + bin_k) % n_bins

                            o_weight = 1.0 - abs(ori_bin - (np.floor(ori_bin) + bin_k))

                            # add weighted contribution to the descriptor
                            descriptor[bin_yi, bin_xi, bin_oi] += weight * y_weight * x_weight * o_weight

        # flatten descriptor and normalize
        flat_desc = descriptor.flatten()
        norm = np.linalg.norm(flat_desc)

        # threshold and renormalize
        if norm > 0:
            flat_desc /= norm
            flat_desc = np.minimum(flat_desc, 0.2)
            norm = np.linalg.norm(flat_desc)
            if norm > 0:
                flat_desc /= norm

        descriptors.append(flat_desc)

    return np.array(descriptors)

def extract_features_from_images(image_dir):
    image_paths = []
    for filename in sorted(os.listdir(image_dir)):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_paths.append(os.path.join(image_dir, filename))

    all_keypoints = []
    all_descriptors = []

    print("[INFO] Extracting features from {} images...".format(len(image_paths)))
    for i, image_path in enumerate(image_paths):
        print(f"Processing image {i+1}/{len(image_paths)}: {os.path.basename(image_path)}")

        img = cv2.imread(image_path)
        keypoints = manual_sift_detector(img)
        descriptors = compute_sift_descriptors(img, keypoints)

        all_keypoints.append(keypoints)
        all_descriptors.append(descriptors)

    return image_paths, all_keypoints, all_descriptors


# Feature matching using ratio test
def match_features(descriptors1, descriptors2, ratio_threshold=0.8):
    tree = cKDTree(descriptors2)

    distances, indices = tree.query(descriptors1, k=2)

    matches = []
    for i in range(len(descriptors1)):
        if len(indices[i]) < 2:
            continue

        if distances[i][0] < ratio_threshold * distances[i][1]:
            matches.append((i, indices[i][0]))

    return matches

def match_features_all_pairs(all_descriptors):
    n_images = len(all_descriptors)
    matches = {}

    print("[INFO] Matching features between all image pairs...")
    for i in range(n_images):
        for j in range(i+1, n_images):
            if len(all_descriptors[i]) == 0 or len(all_descriptors[j]) == 0:
                continue

            print(f"Matching images {i} and {j}...")
            pair_matches = match_features(all_descriptors[i], all_descriptors[j])

            if len(pair_matches) >= 8:  # minimum matches for essential matrix estimation
                matches[(i, j)] = pair_matches

    return matches


# Essential matrix between two views using RANSAC
def compute_essential_matrix(matches, keypoints1, keypoints2, K, ransac_threshold=3.0):
    # matching points
    pts1 = np.float32([keypoints1[match[0]].pt for match in matches])
    pts2 = np.float32([keypoints2[match[1]].pt for match in matches])

    # RANSAC
    E, mask = cv2.findEssentialMat(
        pts1, pts2, K,
        method=cv2.RANSAC,
        prob=0.999,
        threshold=ransac_threshold
    )

    # inliers
    inliers = []
    for i, m in enumerate(mask):
        if m[0] == 1:
            inliers.append(matches[i])

    return E, inliers

def decompose_essential_matrix(E, K, pts1, pts2):
    # extract rotation and translation candidates
    _, R, t, _ = cv2.recoverPose(E, pts1, pts2, K)

    return R, t

def inverse_power_iteration(matrix, num_iters=100, eps=1e-10):
    n = matrix.shape[0]
    b_k = np.random.rand(n)
    b_k = b_k / np.linalg.norm(b_k)

    shift = 1e-3
    I = np.eye(n)

    for _ in range(num_iters):
        try:
            x = np.linalg.solve(matrix + shift * I, b_k)
        except np.linalg.LinAlgError:
            break
        x_norm = np.linalg.norm(x)
        if x_norm < eps:
            break
        b_k = x / x_norm

    return b_k

# Triangulation
def triangulate_points(pts1, pts2, P1, P2):
    # convert points to homogeneous coordinates
    pts1_h = np.hstack((pts1, np.ones((pts1.shape[0], 1))))
    pts2_h = np.hstack((pts2, np.ones((pts2.shape[0], 1))))

    points_3d = []

    for i in range(pts1.shape[0]):
        # DLT matrix for point correspondence
        A = np.zeros((4, 4))

        A[0, :] = pts1_h[i, 0] * P1[2, :] - P1[0, :]
        A[1, :] = pts1_h[i, 1] * P1[2, :] - P1[1, :]
        A[2, :] = pts2_h[i, 0] * P2[2, :] - P2[0, :]
        A[3, :] = pts2_h[i, 1] * P2[2, :] - P2[1, :]

        # SVD
        Vh = inverse_power_iteration(A)
        X = Vh[-1, :] / Vh[-1, -1]  # normalize

        points_3d.append(X[:3])

    return np.array(points_3d)

def get_reprojection_error(X, pts, P):
    # convert to homogeneous coordinates
    X_h = np.hstack((X, np.ones((X.shape[0], 1))))

    # project 3D points back to 2D
    proj = np.dot(X_h, P.T)
    proj = proj[:, :2] / proj[:, 2:3]  # normalize

    # reprojection error
    error = np.sqrt(np.sum((proj - pts)**2, axis=1))

    return error

# # Bundle adjustment
def bundle_adjustment_sparsity(n_cameras, n_points, camera_indices, point_indices):
    m = camera_indices.size * 2  # кожна точка дає 2 рівняння (x, y)
    n = n_cameras * 6 + n_points * 3  # параметри: 6 на камеру, 3 на точку
    A = lil_matrix((m, n), dtype=int)

    for i in range(camera_indices.size):
        cam_idx = camera_indices[i]
        pt_idx = point_indices[i]

        # похідні за 6 параметрами камери
        A[2 * i, cam_idx * 6:cam_idx * 6 + 6] = 1
        A[2 * i + 1, cam_idx * 6:cam_idx * 6 + 6] = 1

        # похідні за 3 координатами точки
        A[2 * i, n_cameras * 6 + pt_idx * 3:n_cameras * 6 + pt_idx * 3 + 3] = 1
        A[2 * i + 1, n_cameras * 6 + pt_idx * 3:n_cameras * 6 + pt_idx * 3 + 3] = 1

    return A

def project_points(params, n_cameras, n_points, camera_indices, point_indices, points_2d, K):
    # extract camera parameters (rotation and translation vectors)
    rotations = params[:n_cameras * 3].reshape(n_cameras, 3)
    translations = params[n_cameras * 3:n_cameras * 6].reshape(n_cameras, 3)

    # extract 3D point positions
    points_3d = params[n_cameras * 6:].reshape(n_points, 3)

    # compute projected points
    projected = np.zeros_like(points_2d)

    for i in range(len(camera_indices)):
        cam_idx = camera_indices[i]
        point_idx = point_indices[i]

        # camera parameters
        if cam_idx >= len(rotations): continue
        rvec = rotations[cam_idx]
        tvec = translations[cam_idx]

        # 3D point
        point = points_3d[point_idx]

        # project using OpenCV
        point_projected, _ = cv2.projectPoints(
            np.array([point]), rvec, tvec, K, None
        )
        projected[i] = point_projected[0, 0]

    return projected.flatten()

def run_bundle_adjustment(cameras, points_3d, points_2d, camera_indices, point_indices, K):
    n_cameras = len(cameras)
    n_points = len(points_3d)

    rotations = []
    translations = []

    for R, t in cameras:
        # convert rotation matrix to rotation vector
        rvec, _ = cv2.Rodrigues(R)
        rotations.append(rvec.flatten())
        translations.append(t.flatten())

    rotations = np.array(rotations)
    translations = np.array(translations)

    params = np.hstack((
        rotations.flatten(),
        translations.flatten(),
        points_3d.flatten()
    ))

    result = least_squares(
        fun=lambda p: project_points(p, n_cameras, n_points, camera_indices, point_indices, points_2d, K) - points_2d.flatten(),
        x0=params,
        jac_sparsity=bundle_adjustment_sparsity(n_cameras, n_points, camera_indices, point_indices),
        verbose=2,
        x_scale='jac',
        ftol=1e-4,
        method='trf'
    )

    # refined parameters
    refined_params = result.x
    refined_rotations = refined_params[:n_cameras * 3].reshape(n_cameras, 3)
    refined_translations = refined_params[n_cameras * 3:n_cameras * 6].reshape(n_cameras, 3)
    refined_points_3d = refined_params[n_cameras * 6:].reshape(n_points, 3)

    # convert rotation vectors back to matrices
    refined_cameras = []
    for i in range(n_cameras):
        R, _ = cv2.Rodrigues(refined_rotations[i])
        t = refined_translations[i]
        refined_cameras.append((R, t))

    return refined_cameras, refined_points_3d


def run_sfm_pipeline(image_dir):
    print("[INFO] Starting feature extraction...")
    image_paths, all_keypoints, all_descriptors = extract_features_from_images(image_dir)

    print("[INFO] Matching features between image pairs...")
    all_matches = match_features_all_pairs(all_descriptors)

    # initialize camera parameters
    img = cv2.imread(image_paths[0])
    h, w = img.shape[:2]
    focal_length = max(h, w)

    K = np.array([
        [focal_length, 0, w/2],
        [0, focal_length, h/2],
        [0, 0, 1]
    ])

    # initialize reconstruction with the first two views
    # find the pair with the most inlier matches
    best_pair = None
    best_inlier_count = 0

    for pair, matches in all_matches.items():
        i, j = pair

        # compute essential matrix
        E, inliers = compute_essential_matrix(
            matches, all_keypoints[i], all_keypoints[j], K
        )

        if len(inliers) > best_inlier_count:
            best_inlier_count = len(inliers)
            best_pair = (i, j, E, inliers)

    if best_pair is None:
        print("[ERROR] No suitable image pair found for initialization")
        return None

    # initialize reconstruction from the best pair
    i, j, E, inliers = best_pair
    print(f"[INFO] Initializing from images {i} and {j} with {len(inliers)} inlier matches")

    # extract matching points for the best pair
    pts1 = np.float32([all_keypoints[i][match[0]].pt for match in inliers])
    pts2 = np.float32([all_keypoints[j][match[1]].pt for match in inliers])

    # decompose essential matrix to get rotation and translation
    R, t = decompose_essential_matrix(E, K, pts1, pts2)

    # define camera matrices
    P1 = np.hstack((np.eye(3), np.zeros((3, 1))))  # first camera at origin
    P2 = np.hstack((R, t))  # second camera with R, t

    # calculate projection matrices
    M1 = np.dot(K, P1)
    M2 = np.dot(K, P2)

    # triangulate points between the initial pair
    points_3d = triangulate_points(pts1, pts2, M1, M2)

    # track which 3D points correspond to which features in each image
    point_tracks = {}
    for idx, (pt_idx1, pt_idx2) in enumerate(inliers):
        point_id = idx
        point_tracks[point_id] = {
            i: pt_idx1,
            j: pt_idx2
        }

    # initialize cameras dictionary
    cameras = {
        i: (np.eye(3), np.zeros((3, 1))),  # First camera at origin
        j: (R, t)  # Second camera with R, t
    }

    # set of reconstructed images
    reconstructed_images = {i, j}

    # incremental reconstruction
    for k in range(len(image_paths)):
        if k in reconstructed_images:
            continue

        print(f"[INFO] Processing image {k}...")

        points_2d = []
        points_3d_idx = []

        for point_id, track in point_tracks.items():
            if k in track:
                # this 3D point is visible in the current image
                kp_idx = track[k]
                points_2d.append(all_keypoints[k][kp_idx].pt)
                points_3d_idx.append(point_id)

        if len(points_2d) < 10:
            print(f"[WARN] Not enough correspondences for image {k}, skipping...")
            continue

        points_2d = np.array(points_2d)
        pts_3d = np.array([points_3d[idx] for idx in points_3d_idx])

        # solve PnP to get the camera pose
        success, rvec, tvec, inliers = cv2.solvePnPRansac(
            pts_3d, points_2d, K, None, iterationsCount=100, reprojectionError=8.0, flags=cv2.SOLVEPNP_EPNP
        )

        if not success or len(inliers) < 10:
            print(f"[WARN] PnP failed for image {k}, skipping...")
            continue

        # convert rotation vector to matrix
        R, _ = cv2.Rodrigues(rvec)
        t = tvec

        # add this camera to the reconstruction
        cameras[k] = (R, t)
        reconstructed_images.add(k)

        # find new matches between this image and already reconstructed images
        for l in reconstructed_images:
            if l == k or (l, k) not in all_matches and (k, l) not in all_matches:
                continue

            # get matches between images k and l
            if (k, l) in all_matches:
                matches = all_matches[(k, l)]
                k_first = True
            else:
                matches = all_matches[(l, k)]
                k_first = False

            # process new matches
            for match in matches:
                if k_first:
                    pt_idx_k, pt_idx_l = match
                else:
                    pt_idx_l, pt_idx_k = match

                # check if the point in l is already tracked
                point_id = None
                for p_id, track in point_tracks.items():
                    if l in track and track[l] == pt_idx_l:
                        point_id = p_id
                        break

                if point_id is not None:
                    # this point is already tracked, add observation in image k
                    if k not in point_tracks[point_id]:
                        point_tracks[point_id][k] = pt_idx_k
                else:
                    # create a new track for this match and triangulate
                    if k_first:
                        pts_k = np.array([all_keypoints[k][pt_idx_k].pt])
                        pts_l = np.array([all_keypoints[l][pt_idx_l].pt])
                    else:
                        pts_k = np.array([all_keypoints[k][pt_idx_k].pt])
                        pts_l = np.array([all_keypoints[l][pt_idx_l].pt])

                    # get camera matrices
                    Pk = np.dot(K, np.hstack((cameras[k][0], cameras[k][1])))
                    Pl = np.dot(K, np.hstack((cameras[l][0], cameras[l][1])))

                    # triangulate this point
                    X = triangulate_points(pts_k, pts_l, Pk, Pl)[0]

                    # add new point to points_3d and create a new track
                    point_id = len(points_3d)
                    points_3d = np.vstack((points_3d, X))
                    point_tracks[point_id] = {
                        k: pt_idx_k,
                        l: pt_idx_l
                    }

    # bundle adjustment data preparation
    camera_indices = []
    point_indices = []
    points_2d_ba = []

    for point_id, track in point_tracks.items():
        for image_id, pt_idx in track.items():
            if image_id in cameras:
                camera_indices.append(image_id)
                point_indices.append(point_id)
                points_2d_ba.append(all_keypoints[image_id][pt_idx].pt)

    camera_indices = np.array(camera_indices)
    point_indices = np.array(point_indices)
    points_2d_ba = np.array(points_2d_ba)

    # convert cameras to a list for bundle adjustment
    camera_list = [cameras[i] for i in sorted(cameras.keys())]

    # run bundle adjustment
    print("[INFO] Running bundle adjustment...")
    refined_cameras, refined_points = run_bundle_adjustment(
        camera_list, points_3d, points_2d_ba, camera_indices, point_indices, K
    )

    # update cameras with refined values
    for i, cam_id in enumerate(sorted(cameras.keys())):
        cameras[cam_id] = refined_cameras[i]

    # return the reconstruction
    reconstruction = {
        'points': refined_points,
        'cameras': cameras,
        'tracks': point_tracks,
        'K': K
    }

    return reconstruction


# def main():
#     print("[INFO] Starting Structure from Motion pipeline...")

#     # Extract features and run SfM manually
#     reconstruction = run_sfm_pipeline(drive_image_dir)

#     if reconstruction is None:
#         print("[ERROR] SfM failed")
#         return

#     # for comparison with pycolmap, also run the official implementation
#     print("\n[INFO] Running pycolmap for comparison...")
#     database_path = os.path.join(local_output_dir, "database.db")

#     # run pycolmap SfM
#     os.makedirs(local_output_dir, exist_ok=True)
#     pycolmap.extract_features(database_path=database_path, image_path=drive_image_dir)
#     pycolmap.match_exhaustive(database_path=database_path)

#     reconstructions = pycolmap.incremental_mapping(
#         database_path=database_path,
#         image_path=drive_image_dir,
#         output_path=local_output_dir
#     )
#     pycolmap_reconstruction = next(iter(reconstructions.values())) if reconstructions else None

#     # extract points from manual reconstruction
#     manual_points = reconstruction['points']
#     manual_colors = np.ones((len(manual_points), 3)) * 0.5

#     # create and save the manual reconstruction point cloud
#     manual_pcd = o3d.geometry.PointCloud()
#     manual_pcd.points = o3d.utility.Vector3dVector(manual_points)
#     manual_pcd.colors = o3d.utility.Vector3dVector(manual_colors)
#     manual_ply_path = os.path.join(drive_output_dir, "manual_reconstruction.ply")
#     o3d.io.write_point_cloud(manual_ply_path, manual_pcd)
#     print(f"[DONE] Manual sparse point cloud saved to: {manual_ply_path}")

#     # pycolmap reconstruction
#     if pycolmap_reconstruction:
#         pycolmap_points = []
#         pycolmap_colors = []
#         for pt in pycolmap_reconstruction.points3D.values():
#             pycolmap_points.append(pt.xyz)
#             pycolmap_colors.append([c / 255.0 for c in pt.color])  # normalize

#         # create Open3D point cloud for pycolmap reconstruction
#         pycolmap_pcd = o3d.geometry.PointCloud()
#         pycolmap_pcd.points = o3d.utility.Vector3dVector(np.array(pycolmap_points))
#         pycolmap_pcd.colors = o3d.utility.Vector3dVector(np.array(pycolmap_colors))
#         pycolmap_ply_path = os.path.join(drive_output_dir, "pycolmap_reconstruction_5.ply")
#         o3d.io.write_point_cloud(pycolmap_ply_path, pycolmap_pcd)
#         print(f"[DONE] Pycolmap sparse point cloud saved to: {pycolmap_ply_path}")

#     print("[DONE] Structure from Motion pipeline completed!")



def main():
    print("[INFO] Starting Structure from Motion pipeline...")

    # Extract features and run SfM manually
    reconstruction = run_sfm_pipeline(drive_image_dir)

    if reconstruction is None:
        print("[ERROR] SfM failed")
        return

    # Extract points from manual reconstruction
    manual_points = reconstruction['points']
    manual_colors = np.ones((len(manual_points), 3)) * 0.5  # grayscale or placeholder colors

    # Create and save the manual reconstruction point cloud
    manual_pcd = o3d.geometry.PointCloud()
    manual_pcd.points = o3d.utility.Vector3dVector(manual_points)
    manual_pcd.colors = o3d.utility.Vector3dVector(manual_colors)
    manual_ply_path = os.path.join(drive_output_dir, "mini_manual_reconstruction.ply")
    o3d.io.write_point_cloud(manual_ply_path, manual_pcd)
    print(f"[DONE] Manual sparse point cloud saved to: {manual_ply_path}")

    print("[DONE] Structure from Motion pipeline completed!")

In [ ]:
main()

[INFO] Starting Structure from Motion pipeline...
[INFO] Starting feature extraction...
[INFO] Extracting features from 60 images...
Processing image 1/60: im_0001.png
Generating base image...
Generating Gaussian kernels...
Generating Gaussian pyramid...
Generating DoG pyramid...
Processing image 2/60: im_0002.png
Generating base image...
Generating Gaussian kernels...
Generating Gaussian pyramid...
Generating DoG pyramid...
Processing image 3/60: im_0003.png
Generating base image...
Generating Gaussian kernels...
Generating Gaussian pyramid...
Generating DoG pyramid...
Processing image 4/60: im_0004.png
Generating base image...
Generating Gaussian kernels...
Generating Gaussian pyramid...
Generating DoG pyramid...
Processing image 5/60: im_0005.png
Generating base image...
Generating Gaussian kernels...
Generating Gaussian pyramid...
Generating DoG pyramid...
Processing image 6/60: im_0006.png
Generating base image...
Generating Gaussian kernels...
Generating Gaussian pyramid...
Gener

In [ ]:
import numpy as np
import cv2
import random
import numpy as np
import cv2
import random

def normalize_points(pts, K):
    pts_h = np.hstack((pts, np.ones((pts.shape[0], 1))))
    pts_norm = (np.linalg.inv(K) @ pts_h.T).T
    return pts_norm[:, :2]

def build_matrix_A(x1, x2):
    A = []
    for i in range(len(x1)):
        u1, v1 = x1[i]
        u2, v2 = x2[i]
        A.append([u1*u2, v1*u2, u2, u1*v2, v1*v2, v2, u1, v1, 1])
    return np.array(A)

def estimate_E_from_8(x1, x2):
    A = build_matrix_A(x1, x2)
    _, _, Vt = np.linalg.svd(A)
    E = Vt[-1].reshape(3, 3)
    U, S, Vt = np.linalg.svd(E)
    S = [1, 1, 0]
    return U @ np.diag(S) @ Vt

def compute_essential_matrix_ransac(matches, keypoints1, keypoints2, K, ransac_threshold=0.005, iterations=1000):
    pts1 = np.float32([keypoints1[m[0]].pt for m in matches])
    pts2 = np.float32([keypoints2[m[1]].pt for m in matches])
    x1 = normalize_points(pts1, K)
    x2 = normalize_points(pts2, K)

    best_inliers = []
    best_E = None

    for _ in range(iterations):
        if len(matches) < 8:
            continue
        idx = np.random.choice(len(matches), 8, replace=False)
        E_candidate = estimate_E_from_8(x1[idx], x2[idx])

        inliers = []
        for i in range(len(x1)):
            x1_h = np.array([x1[i][0], x1[i][1], 1])
            x2_h = np.array([x2[i][0], x2[i][1], 1])
            Ex1 = E_candidate @ x1_h
            Etx2 = E_candidate.T @ x2_h
            error = (x2_h @ E_candidate @ x1_h) ** 2 / (Ex1[0]**2 + Ex1[1]**2 + Etx2[0]**2 + Etx2[1]**2)

            if error < ransac_threshold:
                inliers.append(matches[i])

        if len(inliers) > len(best_inliers):
            best_inliers = inliers
            best_E = E_candidate

    return best_E, best_inliers





def run_sfm_pipeline(image_dir):
    print("[INFO] Starting feature extraction...")
    image_paths, all_keypoints, all_descriptors = extract_features_from_images(image_dir)

    print("[INFO] Matching features between image pairs...")
    all_matches = match_features_all_pairs(all_descriptors)

    # initialize camera parameters
    img = cv2.imread(image_paths[0])
    h, w = img.shape[:2]
    focal_length = max(h, w)

    K = np.array([
        [focal_length, 0, w/2],
        [0, focal_length, h/2],
        [0, 0, 1]
    ])

    # initialize reconstruction with the first two views
    # find the pair with the most inlier matches
    best_pair = None
    best_inlier_count = 0

    for pair, matches in all_matches.items():
        i, j = pair

        # compute essential matrix
        E, inliers = compute_essential_matrix_ransac(
            matches, all_keypoints[i], all_keypoints[j], K
        )

        if len(inliers) > best_inlier_count:
            best_inlier_count = len(inliers)
            best_pair = (i, j, E, inliers)

    if best_pair is None:
        print("[ERROR] No suitable image pair found for initialization")
        return None

    # initialize reconstruction from the best pair
    i, j, E, inliers = best_pair
    print(f"[INFO] Initializing from images {i} and {j} with {len(inliers)} inlier matches")

    # extract matching points for the best pair
    pts1 = np.float32([all_keypoints[i][match[0]].pt for match in inliers])
    pts2 = np.float32([all_keypoints[j][match[1]].pt for match in inliers])

    # decompose essential matrix to get rotation and translation
    R, t = decompose_essential_matrix(E, K, pts1, pts2)

    # define camera matrices
    P1 = np.hstack((np.eye(3), np.zeros((3, 1))))  # first camera at origin
    P2 = np.hstack((R, t))  # second camera with R, t

    # calculate projection matrices
    M1 = np.dot(K, P1)
    M2 = np.dot(K, P2)

    # triangulate points between the initial pair
    points_3d = triangulate_points(pts1, pts2, M1, M2)

    # track which 3D points correspond to which features in each image
    point_tracks = {}
    for idx, (pt_idx1, pt_idx2) in enumerate(inliers):
        point_id = idx
        point_tracks[point_id] = {
            i: pt_idx1,
            j: pt_idx2
        }

    # initialize cameras dictionary
    cameras = {
        i: (np.eye(3), np.zeros((3, 1))),  # First camera at origin
        j: (R, t)  # Second camera with R, t
    }

    # set of reconstructed images
    reconstructed_images = {i, j}

    # incremental reconstruction
    for k in range(len(image_paths)):
        if k in reconstructed_images:
            continue

        print(f"[INFO] Processing image {k}...")

        points_2d = []
        points_3d_idx = []

        for point_id, track in point_tracks.items():
            if k in track:
                # this 3D point is visible in the current image
                kp_idx = track[k]
                points_2d.append(all_keypoints[k][kp_idx].pt)
                points_3d_idx.append(point_id)

        if len(points_2d) < 10:
            print(f"[WARN] Not enough correspondences for image {k}, skipping...")
            continue

        points_2d = np.array(points_2d)
        pts_3d = np.array([points_3d[idx] for idx in points_3d_idx])

        # solve PnP to get the camera pose
        success, rvec, tvec, inliers = cv2.solvePnPRansac(
            pts_3d, points_2d, K, None, iterationsCount=100, reprojectionError=8.0, flags=cv2.SOLVEPNP_EPNP
        )

        if not success or len(inliers) < 10:
            print(f"[WARN] PnP failed for image {k}, skipping...")
            continue

        # convert rotation vector to matrix
        R, _ = cv2.Rodrigues(rvec)
        t = tvec

        # add this camera to the reconstruction
        cameras[k] = (R, t)
        reconstructed_images.add(k)

        # find new matches between this image and already reconstructed images
        for l in reconstructed_images:
            if l == k or (l, k) not in all_matches and (k, l) not in all_matches:
                continue

            # get matches between images k and l
            if (k, l) in all_matches:
                matches = all_matches[(k, l)]
                k_first = True
            else:
                matches = all_matches[(l, k)]
                k_first = False

            # process new matches
            for match in matches:
                if k_first:
                    pt_idx_k, pt_idx_l = match
                else:
                    pt_idx_l, pt_idx_k = match

                # check if the point in l is already tracked
                point_id = None
                for p_id, track in point_tracks.items():
                    if l in track and track[l] == pt_idx_l:
                        point_id = p_id
                        break

                if point_id is not None:
                    # this point is already tracked, add observation in image k
                    if k not in point_tracks[point_id]:
                        point_tracks[point_id][k] = pt_idx_k
                else:
                    # create a new track for this match and triangulate
                    if k_first:
                        pts_k = np.array([all_keypoints[k][pt_idx_k].pt])
                        pts_l = np.array([all_keypoints[l][pt_idx_l].pt])
                    else:
                        pts_k = np.array([all_keypoints[k][pt_idx_k].pt])
                        pts_l = np.array([all_keypoints[l][pt_idx_l].pt])

                    # get camera matrices
                    Pk = np.dot(K, np.hstack((cameras[k][0], cameras[k][1])))
                    Pl = np.dot(K, np.hstack((cameras[l][0], cameras[l][1])))

                    # triangulate this point
                    X = triangulate_points(pts_k, pts_l, Pk, Pl)[0]

                    # add new point to points_3d and create a new track
                    point_id = len(points_3d)
                    points_3d = np.vstack((points_3d, X))
                    point_tracks[point_id] = {
                        k: pt_idx_k,
                        l: pt_idx_l
                    }

    # bundle adjustment data preparation
    camera_indices = []
    point_indices = []
    points_2d_ba = []

    for point_id, track in point_tracks.items():
        for image_id, pt_idx in track.items():
            if image_id in cameras:
                camera_indices.append(image_id)
                point_indices.append(point_id)
                points_2d_ba.append(all_keypoints[image_id][pt_idx].pt)

    camera_indices = np.array(camera_indices)
    point_indices = np.array(point_indices)
    points_2d_ba = np.array(points_2d_ba)

    # convert cameras to a list for bundle adjustment
    camera_list = [cameras[i] for i in sorted(cameras.keys())]

    # run bundle adjustment
    print("[INFO] Running bundle adjustment...")
    refined_cameras, refined_points = run_bundle_adjustment(
        camera_list, points_3d, points_2d_ba, camera_indices, point_indices, K
    )

    # update cameras with refined values
    for i, cam_id in enumerate(sorted(cameras.keys())):
        cameras[cam_id] = refined_cameras[i]

    # return the reconstruction
    reconstruction = {
        'points': refined_points,
        'cameras': cameras,
        'tracks': point_tracks,
        'K': K
    }

    return reconstruction



def main():
    print("[INFO] Starting Structure from Motion pipeline...")

    # Extract features and run SfM manually
    reconstruction = run_sfm_pipeline(drive_image_dir)

    if reconstruction is None:
        print("[ERROR] SfM failed")
        return

    # Extract points from manual reconstruction
    manual_points = reconstruction['points']
    manual_colors = np.ones((len(manual_points), 3)) * 0.5  # grayscale or placeholder colors

    # Create and save the manual reconstruction point cloud
    manual_pcd = o3d.geometry.PointCloud()
    manual_pcd.points = o3d.utility.Vector3dVector(manual_points)
    manual_pcd.colors = o3d.utility.Vector3dVector(manual_colors)
    manual_ply_path = os.path.join(drive_output_dir, "manual_reconstruction_dog.ply")
    o3d.io.write_point_cloud(manual_ply_path, manual_pcd)
    print(f"[DONE] Manual sparse point cloud saved to: {manual_ply_path}")

    print("[DONE] Structure from Motion pipeline completed!")

In [ ]:
main()

[INFO] Starting Structure from Motion pipeline...
[INFO] Starting feature extraction...
[INFO] Extracting features from 60 images...
Processing image 1/60: im_0001.png
Generating base image...
Generating Gaussian kernels...
Generating Gaussian pyramid...
Generating DoG pyramid...
Processing image 2/60: im_0002.png
Generating base image...
Generating Gaussian kernels...
Generating Gaussian pyramid...
Generating DoG pyramid...
Processing image 3/60: im_0003.png
Generating base image...
Generating Gaussian kernels...
Generating Gaussian pyramid...
Generating DoG pyramid...
Processing image 4/60: im_0004.png
Generating base image...
Generating Gaussian kernels...
Generating Gaussian pyramid...
Generating DoG pyramid...
Processing image 5/60: im_0005.png
Generating base image...
Generating Gaussian kernels...
Generating Gaussian pyramid...
Generating DoG pyramid...
Processing image 6/60: im_0006.png
Generating base image...
Generating Gaussian kernels...
Generating Gaussian pyramid...
Gener